# Environment check (W1-T4, owner M)

Sanity-check the training environment before Week 3. This notebook:

1. prints Python / PyTorch / CUDA / GPU info,
2. loads the three encoders (`wav2vec2-base`, `wav2vec2-large-xlsr-53`,
   `wavlm-base`) and runs a forward pass on **1 s of random 16 kHz audio**,
   printing the output shapes, and
3. initialises a **Weights & Biases** project, reading the API key from the
   environment (never hardcoded).

> **On Kaggle/Colab:** enable the GPU accelerator first (Kaggle: Settings ->
> Accelerator -> GPU T4; Colab: Runtime -> Change runtime type -> GPU). Store
> `WANDB_API_KEY` / `HF_TOKEN` as Kaggle *Secrets* or Colab *userdata*, not in
> the notebook. This notebook contains **no pipeline logic** (repo rule 1) --
> it only verifies the environment.


## 1. Environment info

In [ ]:
import platform

import torch

print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print("device :", device)


## 2. Load the three encoders + forward pass on 1 s of random audio

Each model is loaded, run once on random audio, and the `last_hidden_state`
shape is printed, then the model is freed. A failure on one model (e.g. a slow
first-time download of XLSR-53, ~1.2 GB) is caught so the others still report.
Expected: `last_hidden_state = (1, ~49, hidden)` for a 1 s clip
(base hidden = 768, XLSR large hidden = 1024).


In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModel

MODELS = {
    "wav2vec2-base": "facebook/wav2vec2-base",
    "xlsr-53": "facebook/wav2vec2-large-xlsr-53",
    "wavlm-base": "microsoft/wavlm-base",
}

SR = 16000
dummy = torch.randn(SR).numpy()  # 1 second of random audio at 16 kHz

for name, repo in MODELS.items():
    try:
        extractor = AutoFeatureExtractor.from_pretrained(repo)
        model = AutoModel.from_pretrained(repo).to(device).eval()
        inputs = extractor(dummy, sampling_rate=SR, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        shape = tuple(out.last_hidden_state.shape)
        print(f"{name:14s} OK   last_hidden_state={shape}   ({repo})")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
    except Exception as exc:  # noqa: BLE001 - env check reports, does not raise
        print(f"{name:14s} FAILED: {type(exc).__name__}: {exc}")


## 3. Weights & Biases project init

Reads `WANDB_API_KEY` from the environment (`.env`, Kaggle Secrets, or Colab
userdata). If it is absent the cell skips cleanly -- the key is **never** written
into the notebook.


In [ ]:
import os

import wandb

api_key = os.environ.get("WANDB_API_KEY")
if not api_key:
    print("WANDB_API_KEY not set; skipping W&B login.")
    print("Set it as a Kaggle Secret / Colab userdata / .env var -- never hardcode it.")
else:
    wandb.login(key=api_key)  # key comes from the environment only
    run = wandb.init(
        project=os.environ.get("WANDB_PROJECT", "codemix-deepfake-detection"),
        entity=os.environ.get("WANDB_ENTITY"),
        name="env-check",
        job_type="setup",
        mode=os.environ.get("WANDB_MODE", "online"),
    )
    wandb.log({"env_check/ok": 1})
    wandb.finish()
    print("W&B run logged and finished -> project:", run.project)


## Notes

- If all three encoders print an `OK` line with a shape, the modelling stack is
  ready for Week 3.
- On CPU-only machines the forward pass still works (just slower); shapes are
  identical. GPU is only needed for training.
- Record the outcome (GPU model, VRAM, any model that failed to load) in this
  week's log book.
